# CNN Deep Learning Model Training with Comprehensive Evaluation
This notebook trains a Convolutional Neural Network (CNN) model on synthetic foot ulcer risk dataset and evaluates its performance with detailed metrics, confusion matrix, and ROC curve.

## 1. Import Required Libraries

In [ ]:
%pip install tensorflow -q

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, confusion_matrix, classification_report,
                             roc_curve, auc, roc_auc_score)
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Sequential
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import pickle
import json
import os
import warnings

warnings.filterwarnings('ignore')

# Set seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Set style for visualizations
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

ModuleNotFoundError: No module named 'tensorflow'

## 2. Load and Explore Dataset

In [2]:
# Load the synthetic dataset
dataset_path = '../Synthetic_Data/synthetic_foot_ulcer_dataset_RISK.csv'
df = pd.read_csv(dataset_path)

print("Dataset Shape:", df.shape)
print("\nFirst few rows:")
print(df.head())
print("\nDataset Info:")
print(df.info())
print("\nMissing Values:")
print(df.isnull().sum())
print("\nClass Distribution:")
print(df['label'].value_counts())
print("\nClass Distribution (%):")
print(df['label'].value_counts(normalize=True) * 100)

Dataset Shape: (10000, 20)

First few rows:
   temp_heel  press_heel  temp_ball  press_ball  temp_arch  press_arch  \
0  33.498160   58.028572  34.927976   43.946339  32.624075   26.239781   
1  32.931085   23.624257  34.473544   35.298480  35.932924   38.670516   
2  33.564242   27.289444  35.021446   37.006235  32.831767   42.708013   
3  35.861021   44.281370  33.103997   31.850940  40.354115  105.342867   
4  33.483273   46.753650  34.663689   43.651912  33.098887   42.449737   

    temp_toe  press_toe       spo2  heartRate      acc_x     acc_y      acc_z  \
0  32.232334  54.647046  95.621089         81   0.239495 -0.478550   9.866665   
1  35.439762  47.212302  85.198974        119  -6.080159  3.358271  12.137578   
2  32.125253  53.691391  90.927254         73  -2.721560  3.565877  12.156596   
3  33.693606  35.795261  95.935108         92   2.095321  0.520806  11.786854   
4  33.531707  58.868484  95.825943         73  10.116889 -2.761172   8.604085   

       gyro_x      gyro_

## 3. Prepare Data and Split into Train/Test Sets

In [3]:
# Prepare features and target
X = df.drop(['label'], axis=1)
y = df['label']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

# Split data into training (80%) and testing (20%) sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"\nTraining set size: {X_train.shape[0]} samples")
print(f"Testing set size: {X_test.shape[0]} samples")

# Scale features - IMPORTANT for neural networks
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Reshape for CNN1D layers - add a channel dimension
X_train_reshaped = X_train_scaled.reshape((X_train_scaled.shape[0], X_train_scaled.shape[1], 1))
X_test_reshaped = X_test_scaled.reshape((X_test_scaled.shape[0], X_test_scaled.shape[1], 1))

print(f"\nOriginal shape: {X_train_scaled.shape}")
print(f"Reshaped for CNN: {X_train_reshaped.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

Features shape: (10000, 19)
Target shape: (10000,)

Training set size: 8000 samples
Testing set size: 2000 samples

Original shape: (8000, 19)
Reshaped for CNN: (8000, 19, 1)
y_train shape: (8000,)
y_test shape: (2000,)


## 4. Build CNN Model Architecture

In [4]:
# Build the CNN model architecture
cnn_model = Sequential([
    # Input layer
    layers.Input(shape=(19, 1)),
    
    # First Convolutional Block
    layers.Conv1D(filters=64, kernel_size=3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),
    
    # Second Convolutional Block
    layers.Conv1D(filters=128, kernel_size=3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),
    
    # Third Convolutional Block
    layers.Conv1D(filters=256, kernel_size=3, activation='relu', padding='same'),
    layers.BatchNormalization(),
    layers.MaxPooling1D(pool_size=2),
    layers.Dropout(0.3),
    
    # Flatten and Dense layers
    layers.Flatten(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.4),
    
    layers.Dense(128, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    
    # Output layer
    layers.Dense(1, activation='sigmoid')
])

# Compile the model
cnn_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss='binary_crossentropy',
    metrics=['accuracy', keras.metrics.AUC()]
)

# Model summary
print("="*70)
print("CNN MODEL ARCHITECTURE")
print("="*70)
cnn_model.summary()

NameError: name 'Sequential' is not defined

## 5. Train CNN Model

In [ ]:
# Define callbacks
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

# Train the model
print("Training CNN Model...")
print("="*70)

history = cnn_model.fit(
    X_train_reshaped, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    callbacks=[early_stopping, reduce_lr],
    verbose=0
)

print("\n✓ Training Complete!")
print(f"Total epochs trained: {len(history.history['loss'])}")
print(f"Final training loss: {history.history['loss'][-1]:.6f}")
print(f"Final validation loss: {history.history['val_loss'][-1]:.6f}")
print(f"Final training accuracy: {history.history['accuracy'][-1]:.6f}")
print(f"Final validation accuracy: {history.history['val_accuracy'][-1]:.6f}")

## 6. Visualize Training History

In [ ]:
# Plot training and validation metrics
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Loss plot
ax1.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax1.set_ylabel('Loss', fontsize=11, fontweight='bold')
ax1.set_title('Model Loss over Epochs', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Accuracy plot
ax2.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch', fontsize=11, fontweight='bold')
ax2.set_ylabel('Accuracy', fontsize=11, fontweight='bold')
ax2.set_title('Model Accuracy over Epochs', fontsize=12, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nTraining History Summary:")
print(f"Best validation accuracy: {max(history.history['val_accuracy']):.6f}")
print(f"Best validation loss: {min(history.history['val_loss']):.6f}")

## 7. Generate Predictions and Evaluate Performance

In [ ]:
# Generate predictions on test set
y_pred_proba = cnn_model.predict(X_test_reshaped, verbose=0)
y_pred = (y_pred_proba >= 0.5).astype(int).flatten()

# Generate predictions on training set
y_train_pred_proba = cnn_model.predict(X_train_reshaped, verbose=0)
y_train_pred = (y_train_pred_proba >= 0.5).astype(int).flatten()

# Calculate performance metrics
test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)
test_roc_auc = roc_auc_score(y_test, y_pred_proba)

# Training metrics
train_accuracy = accuracy_score(y_train, y_train_pred)
train_precision = precision_score(y_train, y_train_pred)
train_recall = recall_score(y_train, y_train_pred)
train_f1 = f1_score(y_train, y_train_pred)
train_roc_auc = roc_auc_score(y_train, y_train_pred_proba)

# Print all performance metrics
print("="*70)
print("CNN DEEP LEARNING MODEL - PERFORMANCE METRICS")
print("="*70)

print("\n📊 TEST SET PERFORMANCE:")
print("-"*70)
print(f"✓ Accuracy:  {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"✓ Precision: {test_precision:.4f} ({test_precision*100:.2f}%)")
print(f"✓ Recall:    {test_recall:.4f} ({test_recall*100:.2f}%)")
print(f"✓ F1-Score:  {test_f1:.4f}")
print(f"✓ ROC-AUC:   {test_roc_auc:.4f}")

print("\n📊 TRAINING SET PERFORMANCE (for comparison):")
print("-"*70)
print(f"✓ Accuracy:  {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
print(f"✓ Precision: {train_precision:.4f} ({train_precision*100:.2f}%)")
print(f"✓ Recall:    {train_recall:.4f} ({train_recall*100:.2f}%)")
print(f"✓ F1-Score:  {train_f1:.4f}")
print(f"✓ ROC-AUC:   {train_roc_auc:.4f}")

print("\n" + "="*70)
print("DETAILED CLASSIFICATION REPORT (TEST SET)")
print("="*70)
print(classification_report(y_test, y_pred, target_names=['No Risk (0)', 'Risk (1)']))
print("="*70)

## 8. Display Confusion Matrix

In [ ]:
# Create confusion matrix
cm = confusion_matrix(y_test, y_pred)

# Extract values
tn, fp, fn, tp = cm.ravel()

# Plot confusion matrix
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Heatmap
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=['No Risk (0)', 'Risk (1)'],
            yticklabels=['No Risk (0)', 'Risk (1)'],
            cbar_kws={'label': 'Count'},
            ax=ax1,
            annot_kws={'size': 16, 'weight': 'bold'})
ax1.set_ylabel('True Label', fontsize=12, fontweight='bold')
ax1.set_xlabel('Predicted Label', fontsize=12, fontweight='bold')
ax1.set_title('Confusion Matrix - CNN Model', fontsize=14, fontweight='bold')

# Detailed metrics from confusion matrix
specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
sensitivity = tp / (tp + fn) if (tp + fn) != 0 else 0
false_positive_rate = fp / (fp + tn) if (fp + tn) != 0 else 0
false_negative_rate = fn / (fn + tp) if (fn + tp) != 0 else 0

# Create a text summary
metrics_text = f"""
CONFUSION MATRIX BREAKDOWN:

True Negatives (TN):  {tn:4d}  [Correctly predicted No Risk]
True Positives (TP):  {tp:4d}  [Correctly predicted Risk]
False Positives (FP): {fp:4d}  [Incorrectly predicted Risk]
False Negatives (FN): {fn:4d}  [Incorrectly predicted No Risk]

DERIVED METRICS:
Sensitivity (Recall):     {sensitivity:.4f}
Specificity:              {specificity:.4f}
False Positive Rate:      {false_positive_rate:.4f}
False Negative Rate:      {false_negative_rate:.4f}
"""

ax2.text(0.1, 0.5, metrics_text, fontsize=11, family='monospace',
         verticalalignment='center', bbox=dict(boxstyle='round', facecolor='lightgreen', alpha=0.5))
ax2.axis('off')

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("CONFUSION MATRIX ANALYSIS")
print("="*70)
print(metrics_text)

## 9. Plot ROC Curve

In [ ]:
# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
roc_auc_value = auc(fpr, tpr)

# Also calculate for training set
fpr_train, tpr_train, _ = roc_curve(y_train, y_train_pred_proba)
roc_auc_train = auc(fpr_train, tpr_train)

# Plot ROC Curve
plt.figure(figsize=(10, 8))

# Plot test ROC curve
plt.plot(fpr, tpr, color='darkgreen', lw=2.5,
         label=f'Test ROC Curve (AUC = {roc_auc_value:.4f})', marker='o', markersize=4)

# Plot training ROC curve for comparison
plt.plot(fpr_train, tpr_train, color='blue', lw=2.5, linestyle='--',
         label=f'Training ROC Curve (AUC = {roc_auc_train:.4f})', marker='s', markersize=4)

# Plot diagonal reference line (random classifier)
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', label='Random Classifier (AUC = 0.50)')

# Formatting
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate (1 - Specificity)', fontsize=12, fontweight='bold')
plt.ylabel('True Positive Rate (Sensitivity)', fontsize=12, fontweight='bold')
plt.title('ROC Curve - CNN Deep Learning Model\n(Receiver Operating Characteristic Curve)',
          fontsize=14, fontweight='bold')
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)

# Add annotations for key points
plt.scatter([0], [1], marker='*', s=500, color='red', label='Ideal Classifier', zorder=5)

plt.tight_layout()
plt.show()

print("\n" + "="*70)
print("ROC CURVE ANALYSIS")
print("="*70)
print(f"Test Set ROC-AUC Score:     {roc_auc_value:.4f}")
print(f"Training Set ROC-AUC Score: {roc_auc_train:.4f}")
print(f"\nInterpretation:")
print(f"  - AUC = 1.0: Perfect classifier")
print(f"  - AUC = 0.5: Random classifier (diagonal line)")
print(f"  - AUC > 0.5: Better than random")
print(f"  - Current AUC: {roc_auc_value:.4f} - {'Excellent' if roc_auc_value > 0.9 else 'Good' if roc_auc_value > 0.8 else 'Fair'}")
print("="*70)

## 10. Save Trained CNN Model

In [ ]:
# Create models directory if it doesn't exist
models_dir = './models'
if not os.path.exists(models_dir):
    os.makedirs(models_dir)
    print(f"Created models directory: {models_dir}")

# Save the CNN model in multiple formats
# 1. SavedModel format (recommended)
model_path_saved = os.path.join(models_dir, 'cnn_model_saved')
cnn_model.save(model_path_saved)
print(f"✓ CNN Model saved (SavedModel format) to: {model_path_saved}")

# 2. H5 format (alternative)
model_path_h5 = os.path.join(models_dir, 'cnn_model.h5')
cnn_model.save(model_path_h5)
print(f"✓ CNN Model saved (H5 format) to: {model_path_h5}")

# Save the scaler
scaler_path = os.path.join(models_dir, 'cnn_scaler.pkl')
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f"✓ Feature Scaler saved to: {scaler_path}")

# Save model metadata
metadata = {
    'model_type': 'CNN (Convolutional Neural Network)',
    'architecture': 'Conv1D with 3 convolutional blocks + Dense layers',
    'input_shape': (19, 1),
    'test_accuracy': float(test_accuracy),
    'test_precision': float(test_precision),
    'test_recall': float(test_recall),
    'test_f1_score': float(test_f1),
    'test_roc_auc': float(test_roc_auc),
    'train_accuracy': float(train_accuracy),
    'train_roc_auc': float(train_roc_auc),
    'epochs_trained': len(history.history['loss']),
    'batch_size': 32,
    'optimizer': 'Adam',
    'learning_rate': 0.001,
    'feature_names': list(X.columns),
    'n_features': len(X.columns),
    'n_training_samples': len(X_train),
    'n_testing_samples': len(X_test),
}

metadata_path = os.path.join(models_dir, 'cnn_metadata.json')
with open(metadata_path, 'w') as f:
    json.dump(metadata, f, indent=4)
print(f"✓ Model metadata saved to: {metadata_path}")

print("\n" + "="*70)
print("MODEL SAVE SUMMARY")
print("="*70)
print(f"All files saved in: {os.path.abspath(models_dir)}")
print(f"\nSaved Files:")
print(f"  1. cnn_model.h5              - Trained CNN model (H5 format)")
print(f"  2. cnn_model_saved/          - Trained CNN model (SavedModel format)")
print(f"  3. cnn_scaler.pkl            - Feature scaler for preprocessing")
print(f"  4. cnn_metadata.json         - Model metadata and performance info")
print("="*70)

print("\n✅ CNN Model Training and Saving Complete!")
print(f"\nFinal Test Accuracy: {test_accuracy*100:.2f}%")
print(f"Final ROC-AUC Score: {test_roc_auc:.4f}")